# 第 10 章 · MCP（filesystem 示例：agent 侧）

对应书里 § Agent Setup with MCPToolset。书里这份是 `./adk_agent_samples/mcp_agent/agent.py`，
要靠 `adk web` 才有反馈；这里改成能在 notebook 里直接跑，并补了一次真实调用。

与书里的四处差异（都在代码里标了【适配】）：

1. `__file__` 在 notebook 里不存在 → 用 notebook 自身路径（VS Code 的 `__vsc_ipynb_file__`）
2. `MCPToolset` 已弃用 → `McpToolset`，且导入路径改为 `google.adk.tools.mcp_tool`
3. `StdioServerParameters` 要包进 `StdioConnectionParams`（后者多了 timeout）
4. `gemini-2.0-flash` 已下线 → `common.adk_model()`

kernel 选 `Python (agentic-adk)`。服务器是官方 `@modelcontextprotocol/server-filesystem`，走 npx。


In [1]:
# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）
from common import adk_model, quiet_adk_sync_warnings, require_adk, run_once

quiet_adk_sync_warnings()
require_adk()

import os
from pathlib import Path

from google.adk.agents import LlmAgent
from google.adk.tools.mcp_tool import McpToolset, StdioConnectionParams
from mcp.client.stdio import StdioServerParameters

# Create a reliable absolute path to a folder named 'mcp_managed_files'
# 【适配】notebook 没有 __file__：优先用 VS Code 注入的 notebook 路径，退回当前工作目录
ipynb = globals().get("__vsc_ipynb_file__")
_HERE = Path(ipynb).parent if ipynb else Path.cwd()
TARGET_FOLDER_PATH = str(_HERE / "mcp_managed_files")

# Ensure the target directory exists before the agent needs it.
os.makedirs(TARGET_FOLDER_PATH, exist_ok=True)
print("agent 可操作的目录:", TARGET_FOLDER_PATH)

root_agent = LlmAgent(
    model=adk_model(),                      # 【适配】书里写死 gemini-2.0-flash
    name='filesystem_assistant_agent',
    instruction=(
        'Help the user manage their files. You can list files, read files, and write files. '
        f'You are operating in the following directory: {TARGET_FOLDER_PATH}'
    ),
    tools=[
        # 【适配】MCPToolset → McpToolset；StdioServerParameters 现在要包一层
        McpToolset(
            connection_params=StdioConnectionParams(
                server_params=StdioServerParameters(
                    command='npx',
                    args=[
                        "-y",  # Argument for npx to auto-confirm install
                        "@modelcontextprotocol/server-filesystem",
                        # This MUST be an absolute path to a folder.
                        TARGET_FOLDER_PATH,
                    ],
                ),
                timeout=120,            # 【适配】npx 首次要下载包，默认 5s 会超时
            ),
            # Optional: You can filter which tools from the MCP server are exposed.
            # For example, to only allow reading:
            tool_filter=['list_directory', 'read_file', 'read_text_file']
        )
    ],
)


agent 可操作的目录: /home/stvli/agentic_design_patterns/Agentic-Design-Patterns/chapter_notebooks/mcp_managed_files


In [2]:
# 写一个样例文件，再让 agent 通过 MCP 读回来（书里靠 adk web 手动点击，这里自动跑一次）
Path(TARGET_FOLDER_PATH, "sample.txt").write_text("这是给 agent 读的样例文件\n", encoding="utf-8")

finals, _ = run_once(root_agent, "这个目录里有哪些文件？把 sample.txt 的内容读出来。")
for author, text in finals:
    print(f"[{author}] {text}")


[filesystem_assistant_agent] 该目录中包含以下文件：
* **[FILE] sample.txt**

`sample.txt` 的内容为：
```text
这是给 agent 读的样例文件
```
